In [1]:
from databricks.connect import DatabricksSession
from pyspark.sql import functions as F

from src.schemas import ORDERS_SCHEMA, CUSTOMERS_SCHEMA, PAYMENTS_SCHEMA, PRODUCTS_SCHEMA, ORDER_ITEMS_SCHEMA

In [2]:
spark = (
    DatabricksSession.builder
    .serverless()
    .profile("DEFAULT")
    .getOrCreate()
)

In [3]:
spark.sql("""
    SELECT
        current_catalog() AS catalog,
        current_schema() AS schema
""").show()


+---------+-------+
|  catalog| schema|
+---------+-------+
|workspace|default|
+---------+-------+



In [4]:
spark.sql("""
    CREATE SCHEMA IF NOT EXISTS workspace.bronze
""")

DataFrame[]

In [5]:
spark.sql("SHOW SCHEMAS IN workspace").show()

+------------------+
|      databaseName|
+------------------+
|            bronze|
|           default|
|information_schema|
+------------------+



In [6]:
spark.sql("""
    CREATE VOLUME IF NOT EXISTS workspace.bronze.raw_files
""")

DataFrame[]

In [7]:
spark.sql("""
    SHOW VOLUMES IN workspace.bronze
""").show(truncate=False)

+--------+-----------+
|database|volume_name|
+--------+-----------+
|bronze  |raw_files  |
+--------+-----------+



In [8]:
raw_orders_path = "/Volumes/workspace/bronze/raw_files/olist_orders_dataset.csv"

raw_orders = (
    spark.read
    .option("header", True)
    .schema(ORDERS_SCHEMA)
    .csv(raw_orders_path)
)

raw_orders.printSchema()
print("Row count: ", raw_orders.count())

root
 |-- order_id: string (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- order_status: string (nullable = true)
 |-- order_purchase_timestamp: timestamp (nullable = true)
 |-- order_approved_at: timestamp (nullable = true)
 |-- order_delivered_carrier_date: timestamp (nullable = true)
 |-- order_delivered_customer_date: timestamp (nullable = true)
 |-- order_estimated_delivery_date: timestamp (nullable = true)

Row count:  99441


In [9]:
bronze_orders = (
    raw_orders
    .select(
        "*",
        F.col("_metadata.file_path").alias("_source_file"),
        F.col("_metadata.file_modification_time").alias("_source_file_modified_at"),
    )
    .withColumn("_ingested_at", F.current_timestamp())
    .withColumn("_source_entity", F.lit("orders"))
)

bronze_orders.printSchema()

root
 |-- order_id: string (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- order_status: string (nullable = true)
 |-- order_purchase_timestamp: timestamp (nullable = true)
 |-- order_approved_at: timestamp (nullable = true)
 |-- order_delivered_carrier_date: timestamp (nullable = true)
 |-- order_delivered_customer_date: timestamp (nullable = true)
 |-- order_estimated_delivery_date: timestamp (nullable = true)
 |-- _source_file: string (nullable = false)
 |-- _source_file_modified_at: timestamp (nullable = false)
 |-- _ingested_at: timestamp (nullable = false)
 |-- _source_entity: string (nullable = false)



In [10]:
bronze_orders.select(
    "order_id",
    "_source_file",
    "_source_file_modified_at",
    "_ingested_at",
    "_source_entity",
).show(5, truncate=False)

+--------------------------------+-----------------------------------------------------------------+------------------------+--------------------------+--------------+
|order_id                        |_source_file                                                     |_source_file_modified_at|_ingested_at              |_source_entity|
+--------------------------------+-----------------------------------------------------------------+------------------------+--------------------------+--------------+
|e481f51cbdc54678b7cc49136f2d6af7|dbfs:/Volumes/workspace/bronze/raw_files/olist_orders_dataset.csv|2026-08-10 11:57:09     |2026-08-11 11:36:31.741192|orders        |
|53cdb2fc8bc7dce0b6741e2150273451|dbfs:/Volumes/workspace/bronze/raw_files/olist_orders_dataset.csv|2026-08-10 11:57:09     |2026-08-11 11:36:31.741192|orders        |
|47770eb9100c2d0c44946d9cf07ec65d|dbfs:/Volumes/workspace/bronze/raw_files/olist_orders_dataset.csv|2026-08-10 11:57:09     |2026-08-11 11:36:31.741192|orders  

In [11]:
(
    bronze_orders.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("workspace.bronze.orders")
)

In [12]:
spark.sql("""
    DESCRIBE DETAIL workspace.bronze.orders
""").show(truncate=False)

+------+------------------------------------+-----------------------+-----------+--------+-----------------------+-------------------+----------------+-----------------+--------+-----------+---------------------------------------------------------------------------------------------------------------------------------------------------------------------------+----------------+----------------+-----------------------------------------+---------------------------------------------------------------+-------------+
|format|id                                  |name                   |description|location|createdAt              |lastModified       |partitionColumns|clusteringColumns|numFiles|sizeInBytes|properties                                                                                                                                                                 |minReaderVersion|minWriterVersion|tableFeatures                            |statistics                                   

In [13]:
spark.sql("""
    DESCRIBE HISTORY workspace.bronze.orders
""").show(truncate=False)

+-------+-------------------+--------------+-----------------------------+---------------------------------+-------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+----+--------+------------------------------------+------------------------+-----------+-----------------+-------------+----------------------------------------------------------------------------------------------------------------------------------------------------+------------+------------------------------------------------+
|version|timestamp          |userId        |userName                     |operation                        |operationParameters                                                                                                                 

In [14]:
demo = spark.createDataFrame(
    [
        (1, "first"),
        (2, "second"),
        (3, "third"),
    ],
    ["id", "value"]
)

(
    demo.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("workspace.bronze.delta_history_demo")
)

In [15]:
spark.sql("""
    DESCRIBE HISTORY workspace.bronze.delta_history_demo
""").show(truncate=False)

+-------+-------------------+--------------+-----------------------------+---------------------------------+--------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+----+--------+------------------------------------+------------------------+-----------+-----------------+-------------+-------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------

In [16]:
new_row = spark.createDataFrame(
    [(4, "fourth")],
    ["id", "value"]
)

(
    new_row.write
    .format("delta")
    .mode("append")
    .saveAsTable("workspace.bronze.delta_history_demo")
)

In [17]:
spark.sql("""
    DESCRIBE HISTORY workspace.bronze.delta_history_demo
""").show(truncate=False)

+-------+-------------------+--------------+-----------------------------+---------------------------------+--------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+----+--------+------------------------------------+------------------------+-----------+-----------------+-------------+-------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------

In [18]:
spark.sql("""
    SELECT *
    FROM workspace.bronze.delta_history_demo
    VERSION AS OF 0
    ORDER BY id
""").show()

+---+------+
| id| value|
+---+------+
|  1| first|
|  2|second|
|  3| third|
+---+------+



In [19]:
spark.sql("""
    SELECT *
    FROM workspace.bronze.delta_history_demo
    ORDER BY id
""").show()

+---+------+------------+
| id| value|extra_column|
+---+------+------------+
|  1| first|        NULL|
|  2|second|        NULL|
|  3| third|        NULL|
|  4|fourth|        NULL|
+---+------+------------+



In [20]:
bad_schema_df = spark.createDataFrame(
    [
        (5, "fifth", "unexpected")
    ],
    ["id", "value", "extra_column"]
)

(
    bad_schema_df.write
    .format("delta")
    .mode("append")
    .saveAsTable("workspace.bronze.delta_history_demo")
)

In [21]:
spark.sql("""
    ALTER TABLE workspace.bronze.delta_history_demo
    ADD COLUMNS (extra_column STRING)
""")

{"ts": "2026-08-11 14:37:07.866", "level": "ERROR", "logger": "SQLQueryContextLogger", "msg": "[FIELD_ALREADY_EXISTS] Cannot add column, because `extra_column` already exists in \"STRUCT<id: BIGINT, value: STRING, extra_column: STRING>\". SQLSTATE: 42710; line 2 pos 4;\nAddColumns [qualifiedcoltype(None, extra_column, StringType, true, None, None, None, None)]\n+- ResolvedTable com.databricks.sql.managedcatalog.UnityCatalogV2Proxy@772b997, bronze.delta_history_demo, DeltaTableV2(org.apache.spark.sql.classic.SparkSession@4ab74d0f,s3://dbstorage-prod-mxbju/uc/08f86222-fe59-41f7-94e0-296585a7442b/8cdf6fa5-8928-4710-8ae9-cc2034b8497d/__unitystorage/catalogs/9a5290aa-5b91-4822-8ec8-94a528734edf/tables/96d9910f-eee5-4c93-afc6-3d3b4e8a7b6e,Some(CatalogTable(\nCatalog: workspace\nDatabase: bronze\nTable: delta_history_demo\nOwner: branimir.anastassov@gmail.com\nCreated Time: Mon Aug 10 12:14:23 UTC 2026\nLast Access: UNKNOWN\nCreated By: Spark \nType: MANAGED\nProvider: delta\nTable Properties

AnalysisException: [FIELD_ALREADY_EXISTS] Cannot add column, because `extra_column` already exists in "STRUCT<id: BIGINT, value: STRING, extra_column: STRING>". SQLSTATE: 42710; line 2 pos 4;
AddColumns [qualifiedcoltype(None, extra_column, StringType, true, None, None, None, None)]
+- ResolvedTable com.databricks.sql.managedcatalog.UnityCatalogV2Proxy@772b997, bronze.delta_history_demo, DeltaTableV2(org.apache.spark.sql.classic.SparkSession@4ab74d0f,s3://dbstorage-prod-mxbju/uc/08f86222-fe59-41f7-94e0-296585a7442b/8cdf6fa5-8928-4710-8ae9-cc2034b8497d/__unitystorage/catalogs/9a5290aa-5b91-4822-8ec8-94a528734edf/tables/96d9910f-eee5-4c93-afc6-3d3b4e8a7b6e,Some(CatalogTable(
Catalog: workspace
Database: bronze
Table: delta_history_demo
Owner: branimir.anastassov@gmail.com
Created Time: Mon Aug 10 12:14:23 UTC 2026
Last Access: UNKNOWN
Created By: Spark 
Type: MANAGED
Provider: delta
Table Properties: [delta.enableDeletionVectors=true, delta.feature.appendOnly=supported, delta.feature.del

In [ ]:
spark.table(
    "workspace.bronze.delta_history_demo"
).printSchema()

In [ ]:
(
    bad_schema_df.write
    .format("delta")
    .mode("append")
    .saveAsTable("workspace.bronze.delta_history_demo")
)

In [ ]:
spark.sql("""
    SELECT *
    FROM workspace.bronze.delta_history_demo
    ORDER BY id
""").show()

In [ ]:
spark.sql("""
    SELECT
        version,
        timestamp,
        operation,
        readVersion,
        operationMetrics
    FROM (
        DESCRIBE HISTORY workspace.bronze.delta_history_demo
    )
    ORDER BY version DESC
""").show(truncate=False)

In [ ]:
replacement_df = spark.createDataFrame(
    [
        (100, "replacement_a", "new"),
        (200, "replacement_b", "new"),
    ],
    ["id", "value", "extra_column"]
)
replacement_df.show()

In [ ]:
(
    replacement_df.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("workspace.bronze.delta_history_demo")
)

In [ ]:
spark.sql("""
    SELECT *
    FROM workspace.bronze.delta_history_demo
    ORDER BY id
""").show()

In [ ]:
spark.sql("""
    SELECT *
    FROM workspace.bronze.delta_history_demo
    VERSION AS OF 3
    ORDER BY id
""").show()

In [ ]:
updates_df = spark.createDataFrame(
    [
        (100, "replacement_a_updated", "updated"),
        (300, "replacement_c", "new"),
    ],
    ["id", "value", "extra_column"]
)

updates_df.show()

In [ ]:
updates_df.createOrReplaceTempView("updates")

In [ ]:
spark.sql("""
    MERGE INTO workspace.bronze.delta_history_demo AS target
    USING updates AS source
    ON target.id = source.id

    WHEN MATCHED THEN
        UPDATE SET *

    WHEN NOT MATCHED THEN
        INSERT *
""")

In [ ]:
spark.sql("""
    SELECT *
    FROM workspace.bronze.delta_history_demo
    ORDER BY id
""").show(truncate=False)

In [ ]:
spark.sql("""
    SELECT
        version,
        operation,
        readVersion,
        operationMetrics
    FROM (
        DESCRIBE HISTORY workspace.bronze.delta_history_demo
    )
    ORDER BY version DESC
""").show(truncate=False)

In [ ]:
(
    spark.sql("""
        DESCRIBE TABLE EXTENDED workspace.bronze.delta_history_demo
    """)
    .filter("col_name = 'Predictive Optimization'")
    .show(truncate=False)
)


In [ ]:
raw_customers_path = "/Volumes/workspace/bronze/raw_files/olist_customers_dataset.csv"

raw_customers = (
    spark.read
    .option("header", True)
    .schema(CUSTOMERS_SCHEMA)
    .csv(raw_customers_path)
)

raw_customers.printSchema()
print("Rows: ", raw_customers.count())

In [ ]:
bronze_customers = (
    raw_customers
    .select(
        "*",
        F.col("_metadata.file_path").alias("_source_file"),
        F.col("_metadata.file_modification_time")
            .alias("_source_file_modified_at"),
    )
    .withColumn("_ingested_at", F.current_timestamp())
    .withColumn("_source_entity", F.lit("customers"))
)


In [ ]:
bronze_customers.select(
    "customer_id",
    "_source_file",
    "_source_file_modified_at",
    "_ingested_at",
    "_source_entity"
).show(5, truncate=False)

In [ ]:
bronze_customers.write.format("delta").mode("overwrite").saveAsTable("workspace.bronze.customers")

In [ ]:
spark.sql("""
DESCRIBE DETAIL workspace.bronze.customers
""").select(
    "format",
    "name",
    "numFiles",
    "sizeInBytes",
).show(5, truncate=False)

In [22]:
raw_order_items_path = "/Volumes/workspace/bronze/raw_files/olist_order_items_dataset.csv"

raw_order_items = (
    spark.read
    .option("header", True)
    .schema(ORDER_ITEMS_SCHEMA)
    .csv(raw_order_items_path)
)

raw_order_items.printSchema()
print("Rows: ", raw_order_items.count())

root
 |-- order_id: string (nullable = true)
 |-- order_item_id: integer (nullable = true)
 |-- product_id: string (nullable = true)
 |-- seller_id: string (nullable = true)
 |-- shipping_limit_date: timestamp (nullable = true)
 |-- price: decimal(12,2) (nullable = true)
 |-- freight_value: decimal(12,2) (nullable = true)

Rows:  112650


In [23]:
bronze_order_items = (
    raw_order_items
    .select(
        "*",
        F.col("_metadata.file_path").alias("_source_file"),
        F.col("_metadata.file_modification_time")
            .alias("_source_file_modified_at"),
    )
    .withColumn("_ingested_at", F.current_timestamp())
    .withColumn("_source_entity", F.lit("order_items"))
)

In [25]:
bronze_order_items.select(
    "order_id",
    "order_item_id",
    "product_id",
    "_source_file",
    "_ingested_at",
    "_source_entity"
).show(5, truncate=False)

+--------------------------------+-------------+--------------------------------+----------------------------------------------------------------------+--------------------------+--------------+
|order_id                        |order_item_id|product_id                      |_source_file                                                          |_ingested_at              |_source_entity|
+--------------------------------+-------------+--------------------------------+----------------------------------------------------------------------+--------------------------+--------------+
|00010242fe8c5a6d1ba2dd792cb16214|1            |4244733e06e7ecb4970a6e2683c13e61|dbfs:/Volumes/workspace/bronze/raw_files/olist_order_items_dataset.csv|2026-08-11 12:55:17.496335|order_items   |
|00018f77f2f0320c557190d7a144bdd3|1            |e5f2d52b802189ee658865ca93d83a8f|dbfs:/Volumes/workspace/bronze/raw_files/olist_order_items_dataset.csv|2026-08-11 12:55:17.496335|order_items   |
|000229ec398224ef6ca0657d

In [26]:
(
    bronze_order_items.write.
    format("delta")
    .mode("overwrite")
    .saveAsTable("workspace.bronze.order_items")
)

In [28]:
spark.sql("""
DESCRIBE DETAIL workspace.bronze.order_items
""").select(
    "format",
    "name",
    "numFiles",
    "sizeInBytes",
).show(5, truncate=False)

+------+----------------------------+--------+-----------+
|format|name                        |numFiles|sizeInBytes|
+------+----------------------------+--------+-----------+
|delta |workspace.bronze.order_items|1       |3849785    |
+------+----------------------------+--------+-----------+



In [33]:
raw_products_path = "/Volumes/workspace/bronze/raw_files/olist_products_dataset.csv"

raw_products = (
    spark.read
    .option("header", True)
    .schema(PRODUCTS_SCHEMA)
    .csv(raw_products_path)
)

raw_products.printSchema()
print("Rows: ", raw_products.count())

root
 |-- product_id: string (nullable = true)
 |-- product_category_name: string (nullable = true)
 |-- product_name_lenght: integer (nullable = true)
 |-- product_description_lenght: integer (nullable = true)
 |-- product_photos_qty: integer (nullable = true)
 |-- product_weight_g: integer (nullable = true)
 |-- product_length_cm: integer (nullable = true)
 |-- product_height_cm: integer (nullable = true)
 |-- product_width_cm: integer (nullable = true)

Rows:  32951


In [34]:
bronze_products = (
    raw_products
    .select(
        "*",
        F.col("_metadata.file_path").alias("_source_file"),
        F.col("_metadata.file_modification_time")
            .alias("_source_file_modified_at"),
    )
    .withColumn("_ingested_at", F.current_timestamp())
    .withColumn("_source_entity", F.lit("products"))
)

In [36]:
bronze_products.select(
    "product_id",
    "product_category_name",
    "_source_file",
    "_ingested_at",
    "_source_entity"
).show(5, truncate=False)

+--------------------------------+---------------------+-------------------------------------------------------------------+--------------------------+--------------+
|product_id                      |product_category_name|_source_file                                                       |_ingested_at              |_source_entity|
+--------------------------------+---------------------+-------------------------------------------------------------------+--------------------------+--------------+
|1e9e8ef04dbcff4541ed26657ea517e5|perfumaria           |dbfs:/Volumes/workspace/bronze/raw_files/olist_products_dataset.csv|2026-08-11 13:20:01.438677|products      |
|3aa071139cb16b67ca9e5dea641aaa2f|artes                |dbfs:/Volumes/workspace/bronze/raw_files/olist_products_dataset.csv|2026-08-11 13:20:01.438677|products      |
|96bd76ec8810374ed1b65e291975717f|esporte_lazer        |dbfs:/Volumes/workspace/bronze/raw_files/olist_products_dataset.csv|2026-08-11 13:20:01.438677|products      

In [37]:
(
    bronze_products.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("workspace.bronze.products")
)

In [39]:
spark.sql("""
    DESCRIBE DETAIL workspace.bronze.products
""").select(
    "format",
    "name",
    "numFiles",
    "sizeInBytes"
).show(truncate=False)

+------+-------------------------+--------+-----------+
|format|name                     |numFiles|sizeInBytes|
+------+-------------------------+--------+-----------+
|delta |workspace.bronze.products|1       |795790     |
+------+-------------------------+--------+-----------+



In [41]:
raw_payments_path = "/Volumes/workspace/bronze/raw_files/olist_order_payments_dataset.csv"

raw_payments = (
    spark.read
    .option("header", True)
    .schema(PAYMENTS_SCHEMA)
    .csv(raw_payments_path)
)

raw_payments.printSchema()
print("Rows: ", raw_payments.count())

root
 |-- order_id: string (nullable = true)
 |-- payment_sequential: integer (nullable = true)
 |-- payment_type: string (nullable = true)
 |-- payment_installments: integer (nullable = true)
 |-- payment_value: decimal(10,2) (nullable = true)

Rows:  103886


In [42]:
bronze_payments = (
    raw_payments
    .select(
        "*",
        F.col("_metadata.file_path").alias("_source_file"),
        F.col("_metadata.file_modification_time")
            .alias("_source_file_modified_at"),
    )
    .withColumn("_ingested_at", F.current_timestamp())
    .withColumn("_source_entity", F.lit("payments"))
)

In [44]:
bronze_payments.select(
    "order_id",
    "payment_sequential",
    "payment_type",
    "_source_file",
    "_ingested_at",
    "_source_entity"
).show(5, truncate=False)

+--------------------------------+------------------+------------+-------------------------------------------------------------------------+------------------------+--------------+
|order_id                        |payment_sequential|payment_type|_source_file                                                             |_ingested_at            |_source_entity|
+--------------------------------+------------------+------------+-------------------------------------------------------------------------+------------------------+--------------+
|b81ef226f3fe1789b1e8b2acac839d17|1                 |credit_card |dbfs:/Volumes/workspace/bronze/raw_files/olist_order_payments_dataset.csv|2026-08-11 13:23:36.1458|payments      |
|a9810da82917af2d9aefd1278f1dcfa0|1                 |credit_card |dbfs:/Volumes/workspace/bronze/raw_files/olist_order_payments_dataset.csv|2026-08-11 13:23:36.1458|payments      |
|25e8ea4e93396b6fa0d3dd708e76c1bd|1                 |credit_card |dbfs:/Volumes/workspace/bronz

In [45]:
(
    bronze_payments.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("workspace.bronze.payments")
)

In [47]:
spark.sql("""
    DESCRIBE DETAIL workspace.bronze.payments
""").select(
    "format",
    "name",
    "numFiles",
    "sizeInBytes"
).show(truncate=False)

+------+-------------------------+--------+-----------+
|format|name                     |numFiles|sizeInBytes|
+------+-------------------------+--------+-----------+
|delta |workspace.bronze.payments|1       |2060499    |
+------+-------------------------+--------+-----------+



In [48]:
spark.sql("""
CREATE SCHEMA IF NOT EXISTS workspace.silver
""")

DataFrame[]

In [50]:
spark.sql("""
SHOW SCHEMAS IN workspace
""").show(truncate=False)

+------------------+
|databaseName      |
+------------------+
|bronze            |
|default           |
|information_schema|
|silver            |
+------------------+



In [51]:
bronze_orders_df = spark.table("workspace.bronze.orders")

bronze_orders_df.printSchema()
print("Rows: ", bronze_orders_df.count())

root
 |-- order_id: string (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- order_status: string (nullable = true)
 |-- order_purchase_timestamp: timestamp (nullable = true)
 |-- order_approved_at: timestamp (nullable = true)
 |-- order_delivered_carrier_date: timestamp (nullable = true)
 |-- order_delivered_customer_date: timestamp (nullable = true)
 |-- order_estimated_delivery_date: timestamp (nullable = true)
 |-- _source_file: string (nullable = true)
 |-- _source_file_modified_at: timestamp (nullable = true)
 |-- _ingested_at: timestamp (nullable = true)
 |-- _source_entity: string (nullable = true)

Rows:  99441
